In [1]:
import pandas as pd
import numpy as np

In [4]:
from pathlib import Path
main_dir = Path("/u/home/i/iacir21/myscratch")
xlsx_path = main_dir / "regression_replication/IND_KEN_V12.csv"
df_2 = pd.read_csv(xlsx_path)

In [5]:
df_2

,Unnamed: 0,case_id,judges,court_division,court,from_nairobi,IS_APPEAL,length_of_judgement,citation_count,econ_dispute_count,...,respondent_majority_gender_API,applicant_judge_gender_API,respondent_judge_gender_API,applicant_respondent_gender_API,applicant_name,respondent_name,judge_name_x,median_slant,judge_name_y,median_slant_goodvbad
0,0,139782,anthony ndung'u kimani,Family,High Court at Nakuru,0,0,710,0,0,...,UNK,1,0,0,in re h k n (minor),could not be extracted,anthony ndung'u kimani,-0.122863,anthony ndung'u kimani,0.080361
1,3,83775,pauline nyamweya,Land and Environment,High Court at Nairobi (Milimani Law Courts),1,0,972,1,0,...,male,0,0,1,harrison gicharu ng’ang’a,john njoroge murage,pauline nyamweya,0.000650,pauline nyamweya,-0.004394
2,4,652_2,mwangi njoroge,Environment and Land,Environment and Land Court at Kitale,0,0,572,0,0,...,male,1,1,1,Wanyonyi Chekelie,John Masai,mwangi njoroge,-0.113574,NaN,NaN
3,7,141502,margaret waringa muigai,Family,High Court at Nairobi (Milimani Law Courts),1,0,1067,0,0,...,UNK,0,0,0,i re baby j j (child),could not be extracted,margaret waringa muigai,-0.166188,margaret waringa muigai,-0.000100
4,10,49978,joseph kiplagat sergon,NaN,High Court at Mombasa,0,0,186,0,0,...,UNK,1,0,0,in re heinz joachim wahner (deceased),could not be extracted,joseph kiplagat sergon,0.103142,joseph kiplagat sergon,0.057787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35390,159625,135508,rose edwina atieno ougo,Family,High Court at Nairobi (Milimani Law Courts),1,0,714,0,0,...,UNK,0,0,0,in re j (minor),could not be extracted,rose edwina atieno ougo,-0.035262,rose edwina atieno ougo,0.090013
35391,159630,150750,antonina kossy bor,Land and Environment,Environment and Land Court at Nairobi,1,1,1402,3,0,...,male,1,0,0,grace muthoni gichungwa & salome wanjiku mbirwa,samuel kibui mbirwa,antonina kossy bor,-0.158531,NaN,NaN
35392,159631,42263,mary atieno ang'awa,Land and Environment,High Court at Nairobi (Milimani Law Courts),1,0,616,2,0,...,male,1,0,0,wanza ileli,"musyoka kavingo, mutinda kavingo, mwema kaving...",mary atieno ang'awa,0.028084,mary atieno ang'awa,0.132777
35393,159638,122263,stephen kibunja,Land and Environment,Environment and Land Court at Kisii,0,0,734,0,0,...,male,1,1,1,george ombiro obunga,charles ogutu ochiri,stephen kibunja,-0.158049,NaN,NaN


In [5]:
if "judge_name_x" in df_2.columns:
    print("Column exists!")
else:
    print("Column not found.")

Column exists!


In [2]:
df_judges = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/judge_train_caseids.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/u/home/i/iacir21/myscratch/train_test_set/judge_train_caseids.csv'

In [7]:
# Assume df_judges is the judge-level table you showed
df_expanded = (
    df_judges.assign(train_case_ids=df_judges["train_case_ids"].astype(str).str.split(";"))
             .explode("train_case_ids")
             .reset_index(drop=True)
)

# Make sure case IDs are strings (same type as in df_2)
df_expanded["train_case_ids"] = df_expanded["train_case_ids"].str.strip()


In [8]:
cols_to_keep = ["judge_name_x", "length_of_judgement", "case_id"]
df_2_sub = df_2[cols_to_keep].copy()
df_2_sub["case_id"] = df_2_sub["case_id"].astype(str).str.strip()


In [9]:
train_ids = set(df_expanded["train_case_ids"])
df_filtered = df_2_sub[df_2_sub["case_id"].isin(train_ids)].copy()


In [10]:
df_3 = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/median_year_judges.csv")

df_final = df_filtered.merge(
    df_3, 
    on="judge_name_x", 
    how="left"   # keeps all rows in df_filtered, adds matching columns from df_3
)


In [11]:
# Basic save
df_final.to_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_train.csv", index=False)


In [12]:
df_final

,judge_name_x,length_of_judgement,case_id,median_year,eligible_docs,train_docs,test_docs
0,james aaron makau,519,132434,2015.0,786,505,281
1,pauline nyamweya,972,83775,2013.0,1293,660,633
2,lucy nyambura gacheru,2014,98131,2014.0,1039,534,505
3,samson odhiambo okong'o,4317,106493,2013.0,1010,620,390
4,margaret waringa muigai,1067,141502,2014.0,816,432,384
...,...,...,...,...,...,...,...
77731,antonina kossy bor,1402,150750,2014.0,443,247,196
77732,mumbi ngugi,2054,98728,2014.0,667,404,263
77733,teresia mumbua matheka,657,169222,2015.0,468,252,216
77734,stephen kibunja,734,122263,2014.0,843,453,390


In [19]:
num_unique_judges = df_final['judge_name_x'].nunique()
print(f"Number of unique judges: {num_unique_judges}")

Number of unique judges: 180


In [6]:
list_judge_names=['william ouko',
 'benard mweresa eboso',
 'j w mwera j',
 'alnashir ramazanali magan visram',
 'norbury dugdale',
 'w k korir',
 'mary muthoni gitumbi',
 'e k usui macharia',
 'julie oseko',
 'g m njuguna',
 'elija ogoti obaga',
 'none',
 'antony charo mrima',
 "bwonwong'a justus momanyi",
 'r k ondieki',
 'edward nii adjar torgbor',
 'm n gicheru',
 'c obulutsa',
 'olga akech sewe',
 'hellen wasilwa seruya',
 'e malesi',
 't s luvuga',
 'nelly awori matheka',
 'm warsame',
 's k gacheru',
 'monica mbaru',
 'm w wachira',
 'fred kwasi apaloo',
 'thande mugure',
 'ngaah jairus',
 'mohammed khadhar ibrahim',
 'george ernest omondi tunya',
 'm w murage',
 'eugene cotran',
 'felix kombo',
 'edward trevelyan',
 'judith wanjala',
 'david maitai rimita',
 'christine wanjiku meoli',
 'stella ngali mutuku',
 'sharad rao',
 'mwangi njoroge',
 'milicent akinyi odeny',
 'c w githua',
 'washika',
 'mohamed noor kullow',
 's ongeri',
 'mitey j k j',
 'martin muya',
 'e juma',
 'mg mugo j',
 'dolphina alego snr',
 'b m ochoi',
 'mary muhanji kasango',
 'john mutungi',
 'janet nzilani mulwa',
 'h m ng’ang’a',
 'roselyn naliaka nambuye',
 'khapoya s benson',
 'mutitu r m j',
 'm odero',
 'william musya musyoka',
 'r k koech',
 'mwangi',
 'kosgei',
 'g k kimondo',
 'd k matutu esq',
 'r odenyo',
 "kathurima m'inoti",
 'v o adet',
 'stewart mwachiru madzayo',
 'maureen akinyi odero',
 'robert kipkoech limo',
 'abdullah',
 's m mokua',
 'langat-koech c betty',
 'rose edwina atieno ougo',
 'alex george aluri etyang',
 'james rika',
 'raj bahadar bhandari',
 'james aaron makau',
 'pj ransely j',
 'abida ali-aroni',
 'andrew isaac hayanga',
 'sila munyao',
 'nyutu',
 'd n ogoti',
 'grace a mmasi',
 'jackson kasanga mulwa',
 'jonathan bowen havelock',
 'cecilia wathaiya githua',
 'mwai',
 'daniel david ntanda nsereko',
 'hayanga a i',
 'gilbert andalya omwitsa',
 'lawrence peter ouna',
 'betty rashid',
 'akua kuenyehia',
 'jamila mohammed',
 'joseph william alexander butler-sloss',
 'akilano molade akiwumi',
 'enock chirchir cherono',
 'jeanne wanjiku gacheche',
 'daniel kiio musinga',
 'j l a osiemo j',
 'g p omondi',
 'stella munai muketi',
 's k onjoro',
 'wendy k micheni',
 'jaswinder singh sehmi',
 'george martin ongondo',
 'richard mururu mwongo',
 'gurbachan singh pall',
 'linus kassan',
 'amrittal bhagwanji shah',
 'samuel william wako wambuzi',
 'l n mugambi',
 'philip kiptoo tunoi',
 'edward muthoga muriithi',
 'francis gikonyo',
 'muthoga ca',
 'joseph raymond otieno masime',
 'j v o juma',
 'philip nyamu waki',
 's r wewa',
 'a m obura',
 'j munguti',
 'cosmas maundu',
 'r k langat',
 'norbert okumu',
 'lucy njora drsc',
 'chunilal bhagwandas madan',
 'mohammed abdullahi warsame',
 'marete d k njagi',
 'b ombewa',
 'paul kihara kariuki',
 'nicholas randa owano ombija',
 'oscar angote',
 'bernard chunga',
 'apondi m',
 'roseline a oganyo',
 'james onyiego nyarangi',
 "anne apondi ong'injo",
 'said juma chitembwe',
 'robert mugo mutitu',
 'david a onyancha',
 'mandari',
 'hpg wawaru',
 'v karanja',
 'stephen nyangau riechi',
 'john walter onyango otieno',
 'lucy waruguru gitari',
 'c',
 'david christopher porter',
 'ang’awa m a',
 'r n muriuki',
 'timothy o okello',
 'd o onyango',
 'b n olao',
 'hydfbxbas',
 'mathews nderi nduma',
 'l ambasi',
 'charles yano kimutai',
 'benjamin patrick kubo',
 'c k obara',
 'erastus mwaniki githinji',
 'james otieno olola',
 'l k mutai',
 'r nyakundi',
 'osiemo j l a',
 'c a otieno',
 'a a lakha',
 'moijo matayia ole keiwua',
 'y a shikanda',
 'margaret waringa muigai',
 'aggrey otsyula muchelule',
 'abdul majid cockar',
 'r ougo',
 'charles pius chemuttut',
 'of ingutya at wajir',
 'harold grant platt',
 'abigail mshila',
 'rachel biomondo ngetich',
 'antonina kossy bor',
 'jane muyoti onyango',
 'philip john ransley',
 'joyce nuku khaminwa',
 'barabara kiprugut tanui',
 'd a orimba',
 'dermot joseph sheridan',
 'malesi kidali',
 'reuben nyambati nyakundi',
 'loice chepkemoi komingoi',
 'a i hussein',
 'c w meoli',
 'p ngare gesora',
 'kuloba r',
 'susan ndegwa',
 'hpg',
 'teresia mumbua matheka',
 'orenge k i',
 'w j gichimu',
 'jairus ngaah',
 'anthony kaniaru',
 'pritam singh brar',
 'd k njagi marate',
 'lucy nyambura gacheru',
 'd w mburu',
 'david shikomera majanja',
 'nathan shiundu lutta',
 'kaburu bauni',
 'roseline pauline vunoro wendoh',
 'abdulla mustafa',
 'sankale ole kantai',
 'joyce adhiambo aluoch',
 'j k mitey',
 's o mogute',
 'p n maina',
 'g m a ong’ondo',
 'a c a ong’injo',
 'j gandani',
 'j kiarie',
 'b k tanui',
 'l kimaru',
 'jackton boma ojwang',
 'margaret njoki mwangi',
 'daniel kennedy sultani aganyanya',
 'mwera j w j',
 'n njagi',
 'david kenani maraga',
 'byram ongaya',
 't m mwangi',
 'boaz nathan olao',
 'a b shah ja',
 'lydia awino achode',
 'ruth nekoye sitati',
 'kalpana hasmukhrai rawal',
 'justus kituku',
 'c o nyawiri',
 "hedwig imbosa ong'udi",
 'onesmus ndambuthi makau',
 's o oguk',
 'm munyao sila',
 'b p kub0',
 'anne omollo',
 'mathew guy muli',
 'mwilu',
 'j o magori',
 'o a angote',
 's o temu',
 'joseph vitalis odero juma',
 's m mungai',
 'stephen kibunja',
 'ernest fredrick aragon',
 'tutui',
 'musyoka j',
 'b ochieng',
 'w a juma',
 'g n wakahiu',
 'e a obina',
 'florence nyaguthii muchemi',
 'r walekhwa',
 'trail',
 'c k njai',
 'benard ochieng ondego',
 'j k kingori',
 'jacqueline kamau',
 'onguto joseph louis omondi',
 'james n mwaniki',
 'jemutai grace kemei',
 'm o wambani',
 'hellen amolo omondi',
 'charles gitonga mbogo',
 'l m wachira',
 'g w ngenye â€“ macharia',
 'b mosiria',
 'pk rugut',
 'etyang a g a j',
 'james otieno odek',
 'patel v v j',
 'agnes kalekye murgor',
 't a odera',
 'joseph gregory nyamu',
 'lucy waithaka',
 'james patrick trainor',
 'james wakiaga',
 'surrender kumar sachdeva',
 'daniel ogolla',
 'peter john hewett',
 'onesmus kimweli mutungi',
 'richard otieno kwach',
 'christine atieno ochieng',
 'tank j',
 'v k kiptoon',
 'isaac charles cheskaki wambilyangah',
 'j n muniu',
 'sheikh mohammed amin',
 'm m nafula',
 "c o ong'udi",
 "samson odhiambo okong'o",
 'p m muilu',
 't obutu',
 'effie owuor',
 'caroline kemei',
 'francis tuiyott',
 'william mbaya',
 'lesiit jessie w',
 'ndwiga',
 'david kipyegomen kemei',
 'juma j v o',
 'fatuma sichale',
 'd',
 's okongôçöo',
 'e muriuki nyagah',
 'abdulrasul ahmed lakha',
 'cecil henry ethelwood miller',
 "j k ng'arng'ar",
 'm onkoba',
 'luka kiprotich kimaru',
 'john mwangi gachuhi',
 'leonard njagi',
 'mary clausina oundo',
 'p n areri',
 'richard charles namasaka kuloba',
 'esther nyambura maina',
 'james frank shields',
 'zakayo richard chesoni',
 'samwel ndungu mukunya',
 'l n waithaka',
 'm k mwangi',
 'nzioki wa makau',
 'joseph mbalu mutava',
 'm sudi',
 'wilson nkunja kaberia',
 'tanui b k arap j',
 'w kagendo',
 's lamu',
 'fidulhussein esmailji abdullah',
 'nelson jorum abuodha',
 "a g a etyang'",
 'grace lidembu nzioka',
 's k mutai',
 'wilfrida adhiambo okwany',
 'c n ndegwa',
 'mayamba c a',
 'elena g nderitu',
 'mwongo',
 'b r kipyegon',
 'patrick omwenga kiage',
 'patrick j okwaro otieno',
 's n riechi',
 'alister arthur kneller',
 'radido stephen okiyo',
 'riaga samuel cornelius omolo',
 'mumbi ngugi',
 'sarah chibai ondeyo',
 'kunyuk tito',
 'weldon kipyegon korir',
 'george matatia abaleka dulu',
 'muga apondi',
 'kiarie waweru kiarie',
 '0',
 'na',
 'farah s m amin',
 'john wycliffe mwera',
 'j j masiga',
 'george benedict maina kariuki',
 'doreen mulekyo',
 'm wakahora',
 'w k chepseba',
 'lucy ngima mbugua',
 'hazel wandere',
 'j m mutungi',
 'stephen gatembu kairu',
 'derek schofield',
 'linnet ndolo',
 "emmanuel okello o'kubasu",
 'alfred henry simpson',
 'w ouko',
 "john muting'a mativo",
 'peter muchoki njoroge',
 'b j ndeda',
 'george vincent odunga',
 'n',
 's n makila',
 'abdallah j',
 'm amin j',
 'njeri thuku',
 'rewa',
 'k sambu',
 'anthony k mwicigi',
 'william shirley deverell',
 'peter nyagaka areri',
 'johnson evan gicheru',
 "'s' at nairobi)",
 'p j ransley judge',
 "ndung'u h n",
 'pauline nyamweya',
 's m shitubi',
 'eric john ewen law',
 'wanjiru karanja',
 'temba a sitati',
 'john luka osiemo',
 'mulwa j k',
 'maureen atieno onyango',
 'e obaga',
 'eric kennedy okumu ogola',
 'm kasera',
 'aaron g ringera',
 'hilary kiplagat chemitei',
 'martha a nanzushi',
 'gideon p mbito',
 'john amonde mango',
 'ekaterina trendafilova',
 'philomena mbete mwilu',
 'amraphael mbogholi-msagha',
 'lucy mwihaki njuguna',
 'hannah magondi okwengu',
 'khamoni j m j',
 'johnson kiptonui mitey',
 'ezra o awino',
 'e a nyaloti',
 'murugi geteria mugo',
 'kanyi kimondo',
 'samwel odhiambo oguk',
 'lilian nabwire mutende',
 'andayi w francis',
 'abdul rauf samnakay',
 'isaac lenaola',
 "mary atieno ang'awa",
 'roseline lagat-korir',
 'john nyabuto onyiego',
 'john henry sydney todd',
 'm i g moranga',
 'l t lewa',
 'milton stephen asike makhandia',
 'p gichohi',
 'paul kiptenai kimisoy arap birech',
 'alan robin winston hancox',
 'stephen murugu githinji',
 'e wanjala',
 'r n kimingi',
 'e k usui',
 'hatari peter george waweru',
 'felix makoyo',
 'mutungi charles kariuki',
 'tom mbaluto',
 'b mararo',
 'bernard kasavuli',
 's m s soita',
 'joseph kiplagat sergon',
 'martha karambu koome',
 'dorah o chepkwony',
 'cheruto c kipkorir',
 'jessie wanjiku lessit',
 'm mutuku',
 'a k mokoross',
 'n a owino',
 'kenneth d potter',
 'judge',
 'beatrice thuranira jaden',
 'antony ombwayo',
 "anthony ndung'u kimani",
 'joseph raphael karanja',
 'mitei j',
 'samuel elikana ondari bosire',
 'ransley p j',
 'asenath nyaboke ongeri',
 'patrick john kamau',
 'leslie gerald eyre harris',
 'j k sergon',
 'fred andago ochieng',
 's n abuya',
 'e n maina',
 'thripsisa wanjiku cherere',
 'festus azangalala',
 'nagillah chrispin beda',
 'gilbert shikwe',
 'dalmas omondi ohungo',
 'kennedy a bidali',
 'vinubhai vithalbhai patel',
 'william kipsiro tuiyot',
 'pamela mwikali tutui',
 'l n mesa',
 'r o kwach ja',
 'enock chacha mwita',
 'bauni k j',
 'jesse nyagah njagi',
 'p mayova',
 'of kenya',
 "a b mong'are",
 'a k kaniaru',
 'm l nabibya',
 'abdulhalim h athman',
 'roselyne ekirapa aburili',
 'john micheal khamoni',
 'daniel ogola ogembo',
 'alfred mabeya',
 'mathew john anyara emukule',
 'yuvinalis maronga angima',
 't w murigi',
 'kamau pj j',
 'evans w muleka',
 'joel mwaura ngugi']

In [14]:
unique_judges_df = set(df_final["judge_name_x"].unique())
unique_judges_list = set(list_judge_names)


In [15]:
# in list_judge_names but not in df_judges["judge(s)"]
only_in_list = unique_judges_list - unique_judges_df  

# in df_judges["judge(s)"] but not in list_judge_names
only_in_df = unique_judges_df - unique_judges_list  

# intersection (common judges in both)
common = unique_judges_list & unique_judges_df  


In [20]:
len(common)

179

In [21]:
num_unique_judges = df_final['judge_name_x'].nunique()
print(f"Number of unique judges: {num_unique_judges}")

Number of unique judges: 180


In [24]:
num_unique_judges = df_2['judge_name_x'].nunique()
print(f"Number of unique judges: {num_unique_judges}")

Number of unique judges: 180


In [16]:

print("Only in list_judge_names:", only_in_list)

Only in list_judge_names: {'james frank shields', 'l n waithaka', 'a k mokoross', 'c obulutsa', 'mutitu r m j', 'j gandani', 'abdul rauf samnakay', 'p n areri', "a b mong'are", "c o ong'udi", 'e k usui', 'r k koech', 'e n maina', 'james n mwaniki', 'm amin j', 'mwilu', 'gideon p mbito', 'gurbachan singh pall', 'gilbert shikwe', 'c w githua', 's o mogute', 'a m obura', 'linus kassan', 'e muriuki nyagah', 'c w meoli', 'ernest fredrick aragon', 'mwera j w j', 'stewart mwachiru madzayo', 'malesi kidali', 'c o nyawiri', 's r wewa', 'of ingutya at wajir', 'felix makoyo', 'leslie gerald eyre harris', 'richard otieno kwach', 'etyang a g a j', 'langat-koech c betty', 'nyutu', 'james otieno odek', 'j k sergon', 'susan ndegwa', 'wilson nkunja kaberia', 'eric john ewen law', 'm w murage', 'peter john hewett', 'mwongo', 't a odera', 'musyoka j', 'hpg', 'c', 'alan robin winston hancox', 'd k matutu esq', 'e juma', 'm m nafula', 'rachel biomondo ngetich', 'edward nii adjar torgbor', 't w murigi', 'm 

In [17]:
print("Only in df_judges['judge(s)']:", only_in_df)

Only in df_judges['judge(s)']: {'g w ngenye macharia'}


In [18]:
print("Common:", common)

Common: {'pauline nyamweya', 'onesmus kimweli mutungi', 'philip john ransley', 'david a onyancha', 'antony charo mrima', 'christine atieno ochieng', 'isaac lenaola', 'jackton boma ojwang', 'jessie wanjiku lessit', 'jane muyoti onyango', 'mary muhanji kasango', 'mg mugo j', 'teresia mumbua matheka', 'olga akech sewe', 'roselyne ekirapa aburili', "anne apondi ong'injo", 'radido stephen okiyo', 'philip nyamu waki', 'eric kennedy okumu ogola', 'kanyi kimondo', 'monica mbaru', 'james aaron makau', 'peter muchoki njoroge', 'lucy waruguru gitari', 'david shikomera majanja', 'alnashir ramazanali magan visram', 'alex george aluri etyang', 'tom mbaluto', 'george benedict maina kariuki', 'fred andago ochieng', 'james otieno olola', 'jesse nyagah njagi', 'cecilia wathaiya githua', 'john nyabuto onyiego', 'john luka osiemo', 'nzioki wa makau', 'hilary kiplagat chemitei', 'charles yano kimutai', 'hannah magondi okwengu', 'lydia awino achode', 'anne omollo', 'mathews nderi nduma', 'richard mururu mwo

In [22]:
root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_eligible"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"

In [23]:
import os
from tqdm import tqdm

In [23]:
for judge in tqdm(list_judge_names):
    
    df_temp = df_final[df_final["judge_name_x"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
            
        
        f = open(root_path,"r").read()
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        g = open(file_path,"w+")
        g.write(f)
        
        

100% 504/504 [10:59<00:00,  1.31s/it]


In [24]:
# Count how many unique normalized_judge values exist
num_unique_judges = df_final["normalized_judges"].nunique()
print("Number of unique normalized judges:", num_unique_judges)


Number of unique normalized judges: 579


In [26]:
import os
import pandas as pd

base_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"   # replace with your actual path

judge_data = []

# Loop over each judge folder
for judge_folder in os.listdir(base_dir):
    judge_path = os.path.join(base_dir, judge_folder)
    
    if os.path.isdir(judge_path):
        total_words = 0
        num_cases_v2 = 0
        
        # Loop over each .txt file inside the judge's folder
        for file_name in os.listdir(judge_path):
            if file_name.endswith(".txt"):
                file_path = os.path.join(judge_path, file_name)
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                    total_words += len(text.split())
                    num_cases_v2 += 1
        
        judge_data.append([judge_folder, total_words, num_cases_v2])

# Convert to DataFrame
df_tokens = pd.DataFrame(judge_data, columns=["judge_name", "num_tokens", "num_cases_v2"])

print(df_tokens.head())


        judge_name  num_tokens  num_cases_v2
0  's' at nairobi)        3003             1
1                0        1986             1
2        a a lakha         216             2
3     a b mong'are         317             1
4      a b shah ja        6261            11


In [27]:
df_tokens

,judge_name,num_tokens,num_cases_v2
0,'s' at nairobi),3003,1
1,0,1986,1
2,a a lakha,216,2
3,a b mong'are,317,1
4,a b shah ja,6261,11
...,...,...,...
494,william shirley deverell,48533,34
495,wilson nkunja kaberia,7141,1
496,y a shikanda,16114,6
497,yuvinalis maronga angima,524763,259


In [28]:
df_final

,normalized_judges,length_of_judgement,case_id,median_year,eligible_docs,train_docs
0,james aaron makau,519,132434,2015.0,786,505
1,pauline nyamweya,972,83775,2013.0,1293,660
2,lucy nyambura gacheru,2014,98131,2014.0,1039,534
3,samson odhiambo okong'o,4317,106493,2013.0,1010,620
4,margaret waringa muigai,1067,141502,2014.0,816,432
...,...,...,...,...,...,...
82258,antonina kossy bor,1402,150750,2014.0,443,247
82259,mumbi ngugi,2054,98728,2014.0,667,404
82260,teresia mumbua matheka,657,169222,2015.0,468,252
82261,stephen kibunja,734,122263,2014.0,843,453


In [29]:
df_thresh = df_tokens[df_tokens["num_tokens"]>=50000]

In [30]:
len(df_tokens[df_tokens["num_tokens"]>=250000])

154

In [31]:
df_thresh

,judge_name,num_tokens,num_cases_v2
7,a i hussein,50302,33
11,aaron g ringera,76309,32
19,abida ali-aroni,418598,354
20,abigail mshila,431966,292
21,aggrey otsyula muchelule,658164,751
...,...,...,...
489,wilfrida adhiambo okwany,883215,428
490,william kipsiro tuiyot,74072,50
492,william musya musyoka,1746719,1211
493,william ouko,465681,476


In [32]:
df_thresh.to_csv("/u/home/i/iacir21/myscratch/replication/df_thresh_train_via.csv", index=False)


In [ ]:
***************************TRAIN**************************

In [27]:
import pandas as pd

# --- 1. Load data ---
df_eligible = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/train_case_ids_norm.csv")
df_medians = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/median_year_judges_vnorm.csv")

cols_to_keep = ["judge_name_y", "length_of_judgement", "case_id"]
df_final = df_2[cols_to_keep].copy()

# --- 2. Filter df_final to keep only eligible cases ---
df_filtered = df_final[df_final["case_id"].isin(df_eligible["case_id"])].copy()

# --- 3. Merge median year info ---
df_filtered = df_filtered.merge(
    df_medians,
    on="judge_name_y",
    how="left"
)

# --- 4. Save output ---
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_train.csv"
df_filtered.to_csv(output_path, index=False)

print(f"✅ Saved eligible judge-level file to:\n{output_path}")
print(f"Rows kept: {len(df_filtered)} / {len(df_final)} total")

# --- 5. Compare judge lists ---
unique_judges_in_filtered = set(df_filtered["judge_name_y"].unique())
unique_judges_in_list = set(list_judge_names)

only_in_list = unique_judges_in_list - unique_judges_in_filtered
only_in_df   = unique_judges_in_filtered - unique_judges_in_list
common       = unique_judges_in_list & unique_judges_in_filtered

print("Only in list_judge_names:", only_in_list)
print("Only in filtered df:", only_in_df)
print("Common:", common)

✅ Saved eligible judge-level file to:
/u/home/i/iacir21/myscratch/replication/judge_normalized_train.csv
Rows kept: 16193 / 35395 total
Only in list_judge_names: {'stephen kibunja', 'pk rugut', 'm l nabibya', 's r wewa', 'wilson nkunja kaberia', 's ongeri', 'c n ndegwa', 'b k tanui', 'orenge k i', 'j v o juma', 'cecil henry ethelwood miller', "'s' at nairobi)", 'elena g nderitu', 'temba a sitati', 'r k ondieki', 'm amin j', 'e obaga', 'm wakahora', 'martha a nanzushi', 'mg mugo j', 'norbert okumu', 'etyang a g a j', 'charles gitonga mbogo', 'joseph raymond otieno masime', "a b mong'are", 'stewart mwachiru madzayo', 'jackson kasanga mulwa', 'e malesi', 'l k mutai', 'e muriuki nyagah', 'jane muyoti onyango', 'philip john ransley', 'b p kub0', 'j k sergon', 'kuloba r', 'justus kituku', 'l m wachira', 'john henry sydney todd', 'riaga samuel cornelius omolo', 'raj bahadar bhandari', 'milicent akinyi odeny', 'j kiarie', 'gurbachan singh pall', 's n riechi', 'j m mutungi', 'b m ochoi', 'james

In [28]:
len(common)

141

In [29]:
print("Common:", common)

Common: {'thande mugure', 'jemutai grace kemei', 'lucy ngima mbugua', 'joel mwaura ngugi', 'alnashir ramazanali magan visram', 'beatrice thuranira jaden', 'martha karambu koome', 'jonathan bowen havelock', 'na', 'george vincent odunga', 'edward muthoga muriithi', 'jessie wanjiku lessit', 'said juma chitembwe', 'roseline lagat-korir', 'byram ongaya', 'daniel kennedy sultani aganyanya', 'david kipyegomen kemei', 'david shikomera majanja', 'nelson jorum abuodha', 'richard mururu mwongo', 'john wycliffe mwera', 'james rika', 'reuben nyambati nyakundi', "anthony ndung'u kimani", "samson odhiambo okong'o", 'kalpana hasmukhrai rawal', 'lesiit jessie w', 'onesmus kimweli mutungi', 'farah s m amin', 'charles yano kimutai', 'patrick j okwaro otieno', 'stella ngali mutuku', 'amraphael mbogholi-msagha', 'john nyabuto onyiego', 'fred andago ochieng', 'james aaron makau', 'john micheal khamoni', 'erastus mwaniki githinji', "hedwig imbosa ong'udi", 'kanyi kimondo', 'radido stephen okiyo', 'thripsisa 

In [32]:
shutil.rmtree("/u/home/i/iacir21/myscratch/replication/judges_corpus_train", ignore_errors=True)

In [33]:
root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_train_vnorm"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"

import os
from tqdm import tqdm

for judge in tqdm(list_judge_names):
    
    df_temp = df_filtered[df_filtered["judge_name_y"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
            
        
        f = open(root_path,"r").read()
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        g = open(file_path,"w+")
        g.write(f)
        
# Count how many unique normalized_judge values exist
num_unique_judges = df_final["judge_name_y"].nunique()
print("Number of unique normalized judges:", num_unique_judges)

100% 504/504 [01:27<00:00,  5.78it/s]

Number of unique normalized judges: 141


In [ ]:
import os
from tqdm import tqdm
import pandas as pd

root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_train"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"
df_filtered=pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_train_v1.csv")
# ============ CONFIG: SET YOUR STARTING JUDGE HERE ============
RESUME_FROM_JUDGE = "yuvinalis maronga angima"  # Replace with actual judge name
SKIP_MODE = True  # Set to False if you want to start fresh from this judge
# ==============================================================

started = False  # Flag to track when to start processing

for judge in tqdm(list_judge_names):
    
    # Skip until we reach the resume point
    if SKIP_MODE:
        if not started:
            if judge == RESUME_FROM_JUDGE:
                started = True
                print(f"\n✅ Starting from judge: {judge}\n")
            else:
                continue  # Skip this judge
    
    df_temp = df_filtered[df_filtered["judge_name_x"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    print(f"Processing judge: {judge} ({len(list_ids)} cases)")
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        # Skip if file already exists (resume protection)
        if os.path.exists(file_path):
            continue
            
        try:
            with open(root_path, "r") as f:
                content = f.read()
            
            with open(file_path, "w+") as g:
                g.write(content)
        except Exception as e:
            print(f"Error processing {case_id} for {judge}: {e}")
            continue

print("\n✅ Processing complete!")

# Count how many unique judges
num_unique_judges = df_filtered["judge_name_x"].nunique()
print(f"Number of unique judges: {num_unique_judges}")

In [34]:
import os

# Method 1: Simple and clean (recommended)
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"
num_folders = len([name for name in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, name))])
print(f"Number of folders: {num_folders}")

Number of folders: 141


In [35]:
import os
import pandas as pd

# ============ Step 1: Generate token counts (your existing code) ============
base_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"
judge_data = []

for judge_folder in os.listdir(base_dir):
    judge_path = os.path.join(base_dir, judge_folder)
    
    if os.path.isdir(judge_path):
        total_words = 0
        num_cases_v2 = 0
        
        for file_name in os.listdir(judge_path):
            if file_name.endswith(".txt"):
                file_path = os.path.join(judge_path, file_name)
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                    total_words += len(text.split())
                    num_cases_v2 += 1
        
        judge_data.append([judge_folder, total_words, num_cases_v2])

df_tokens = pd.DataFrame(judge_data, columns=["judge_name", "num_tokens", "num_cases_v2"])

# ============ Step 2: Load existing CSV ============
df_existing = pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_train.csv")

# ============ Step 3: Merge the two dataframes ============
df_merged = df_existing.merge(
    df_tokens,
    left_on="judge_name_y",
    right_on="judge_name",
    how="left"
)

# Drop the duplicate judge_name column
df_merged = df_merged.drop(columns=["judge_name"])

# ============ Step 4: Comparison checks ============
print("=== Merged DataFrame Info ===")
print(df_merged.head())
print(f"\nTotal rows: {len(df_merged)}")
print(f"Rows with token data: {df_merged['num_tokens'].notna().sum()}")
print(f"Rows missing token data: {df_merged['num_tokens'].isna().sum()}")

# Check if case counts match
print("\n=== Case Count Comparison ===")
# Group by judge to compare eligible_docs vs num_cases_v2
judge_comparison = df_merged.groupby('judge_name_y').agg({
    'eligible_docs': 'first',
    'num_cases_v2': 'first'
}).reset_index()

judge_comparison['match'] = judge_comparison['eligible_docs'] == judge_comparison['num_cases_v2']
mismatches = judge_comparison[~judge_comparison['match']]

print(f"Judges with matching case counts: {judge_comparison['match'].sum()}")
print(f"Judges with mismatched case counts: {(~judge_comparison['match']).sum()}")

if len(mismatches) > 0:
    print("\nMismatched judges:")
    print(mismatches)

# Check token threshold
df_merged['meets_threshold'] = df_merged['num_tokens'] >= 50000
print(f"\n=== Token Threshold (>=50,000) ===")
print(f"Judges meeting threshold: {df_merged['meets_threshold'].sum()}")
print(f"Judges below threshold: {(~df_merged['meets_threshold']).sum()}")

# ============ Step 5: Save merged file ============
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_train_merged.csv"
df_merged.to_csv(output_path, index=False)
print(f"\n✅ Merged file saved to: {output_path}")

# ============ Step 6: Save filtered version (threshold >= 50k) ============
df_thresh = df_merged[df_merged['num_tokens'] >= 50000].copy()
thresh_output = "/u/home/i/iacir21/myscratch/replication/judge_normalized_train_threshold_50k.csv"
df_thresh.to_csv(thresh_output, index=False)
print(f"✅ Threshold file saved to: {thresh_output}")
print(f"   Judges included: {len(df_thresh['judge_name_y'].unique())}")


=== Merged DataFrame Info ===
             judge_name_y  length_of_judgement case_id  median_year  \
0  anthony ndung'u kimani                  710  139782       2017.0   
1        pauline nyamweya                  972   83775       2015.0   
2  joseph kiplagat sergon                  186   49978       2013.0   
3             anne omollo                  964  117626       2017.0   
4    dalmas omondi ohungo                  629  146164       2018.0   

   eligible_docs  train_docs  test_docs  num_tokens  num_cases_v2  
0            153          79         74      123640            79  
1            294         172        122      323382           172  
2            777         412        365      366500           412  
3            385         249        136      296236           249  
4            240         149         91      201738           149  

Total rows: 16193
Rows with token data: 16193
Rows missing token data: 0

=== Case Count Comparison ===
Judges with matching case coun

In [ ]:
***************************TEST**************************

In [36]:
import pandas as pd

# --- 1. Load data ---
df_eligible = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/test_case_ids_vnorm.csv")
df_medians = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/median_year_judges_vnorm.csv")

cols_to_keep = ["judge_name_y", "length_of_judgement", "case_id"]
df_final = df_2[cols_to_keep].copy()

# --- 2. Filter df_final to keep only eligible cases ---
df_filtered = df_final[df_final["case_id"].isin(df_eligible["case_id"])].copy()

# --- 3. Merge median year info ---
df_filtered = df_filtered.merge(
    df_medians,
    on="judge_name_y",
    how="left"
)

# --- 4. Save output ---
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_test.csv"
df_filtered.to_csv(output_path, index=False)

print(f"✅ Saved eligible judge-level file to:\n{output_path}")
print(f"Rows kept: {len(df_filtered)} / {len(df_final)} total")

# --- 5. Compare judge lists ---
unique_judges_in_filtered = set(df_filtered["judge_name_y"].unique())
unique_judges_in_list = set(list_judge_names)

only_in_list = unique_judges_in_list - unique_judges_in_filtered
only_in_df   = unique_judges_in_filtered - unique_judges_in_list
common       = unique_judges_in_list & unique_judges_in_filtered

print("Only in list_judge_names:", only_in_list)
print("Only in filtered df:", only_in_df)
print("Common:", common)

✅ Saved eligible judge-level file to:
/u/home/i/iacir21/myscratch/replication/judge_normalized_test.csv
Rows kept: 10764 / 35395 total
Only in list_judge_names: {'stephen kibunja', 'pk rugut', 'm l nabibya', 's r wewa', 'wilson nkunja kaberia', 's ongeri', 'c n ndegwa', 'b k tanui', 'orenge k i', 'j v o juma', 'cecil henry ethelwood miller', "'s' at nairobi)", 'elena g nderitu', 'temba a sitati', 'r k ondieki', 'm amin j', 'e obaga', 'm wakahora', 'martha a nanzushi', 'mg mugo j', 'norbert okumu', 'etyang a g a j', 'charles gitonga mbogo', 'joseph raymond otieno masime', "a b mong'are", 'stewart mwachiru madzayo', 'jackson kasanga mulwa', 'e malesi', 'l k mutai', 'e muriuki nyagah', 'jane muyoti onyango', 'philip john ransley', 'b p kub0', 'j k sergon', 'kuloba r', 'justus kituku', 'l m wachira', 'john henry sydney todd', 'riaga samuel cornelius omolo', 'raj bahadar bhandari', 'milicent akinyi odeny', 'j kiarie', 'gurbachan singh pall', 's n riechi', 'j m mutungi', 'b m ochoi', 'james 

In [37]:
len(common)

141

In [38]:
print("Common:", common)

Common: {'thande mugure', 'lucy ngima mbugua', 'jemutai grace kemei', 'joel mwaura ngugi', 'alnashir ramazanali magan visram', 'beatrice thuranira jaden', 'jonathan bowen havelock', 'martha karambu koome', 'na', 'george vincent odunga', 'edward muthoga muriithi', 'jessie wanjiku lessit', 'said juma chitembwe', 'roseline lagat-korir', 'byram ongaya', 'david kipyegomen kemei', 'nelson jorum abuodha', 'david shikomera majanja', 'daniel kennedy sultani aganyanya', 'richard mururu mwongo', 'james rika', 'john wycliffe mwera', 'reuben nyambati nyakundi', "anthony ndung'u kimani", "samson odhiambo okong'o", 'kalpana hasmukhrai rawal', 'lesiit jessie w', 'onesmus kimweli mutungi', 'farah s m amin', 'charles yano kimutai', 'patrick j okwaro otieno', 'stella ngali mutuku', 'amraphael mbogholi-msagha', 'john nyabuto onyiego', 'fred andago ochieng', 'james aaron makau', 'erastus mwaniki githinji', 'john micheal khamoni', "hedwig imbosa ong'udi", 'kanyi kimondo', 'radido stephen okiyo', 'thripsisa 

In [39]:
root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_test_vnorm"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_test"

import os
from tqdm import tqdm

for judge in tqdm(list_judge_names):
    
    df_temp = df_filtered[df_filtered["judge_name_y"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
            
        
        f = open(root_path,"r").read()
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        g = open(file_path,"w+")
        g.write(f)
        
# Count how many unique normalized_judge values exist
num_unique_judges = df_final["judge_name_y"].nunique()
print("Number of unique normalized judges:", num_unique_judges)

100% 504/504 [01:01<00:00,  8.15it/s]

Number of unique normalized judges: 141


In [ ]:
import os
from tqdm import tqdm
import pandas as pd

root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_test"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_test"
df_filtered=pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_test_v1.csv")
# ============ CONFIG: SET YOUR STARTING JUDGE HERE ============
RESUME_FROM_JUDGE = "yuvinalis maronga angima"  # Replace with actual judge name
SKIP_MODE = True  # Set to False if you want to start fresh from this judge
# ==============================================================

started = False  # Flag to track when to start processing

for judge in tqdm(list_judge_names):
    
    # Skip until we reach the resume point
    if SKIP_MODE:
        if not started:
            if judge == RESUME_FROM_JUDGE:
                started = True
                print(f"\n✅ Starting from judge: {judge}\n")
            else:
                continue  # Skip this judge
    
    df_temp = df_filtered[df_filtered["judge_name_x"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    print(f"Processing judge: {judge} ({len(list_ids)} cases)")
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        # Skip if file already exists (resume protection)
        if os.path.exists(file_path):
            continue
            
        try:
            with open(root_path, "r") as f:
                content = f.read()
            
            with open(file_path, "w+") as g:
                g.write(content)
        except Exception as e:
            print(f"Error processing {case_id} for {judge}: {e}")
            continue

print("\n✅ Processing complete!")

# Count how many unique judges
num_unique_judges = df_filtered["judge_name_x"].nunique()
print(f"Number of unique judges: {num_unique_judges}")

In [40]:
import os

# Method 1: Simple and clean (recommended)
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_test"
num_folders = len([name for name in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, name))])
print(f"Number of folders: {num_folders}")

Number of folders: 141


In [41]:
import os
import pandas as pd

# ============ Step 1: Generate token counts (your existing code) ============
base_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_test"
judge_data = []

for judge_folder in os.listdir(base_dir):
    judge_path = os.path.join(base_dir, judge_folder)
    
    if os.path.isdir(judge_path):
        total_words = 0
        num_cases_v2 = 0
        
        for file_name in os.listdir(judge_path):
            if file_name.endswith(".txt"):
                file_path = os.path.join(judge_path, file_name)
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                    total_words += len(text.split())
                    num_cases_v2 += 1
        
        judge_data.append([judge_folder, total_words, num_cases_v2])

df_tokens = pd.DataFrame(judge_data, columns=["judge_name", "num_tokens", "num_cases_v2"])

# ============ Step 2: Load existing CSV ============
df_existing = pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_test.csv")

# ============ Step 3: Merge the two dataframes ============
df_merged = df_existing.merge(
    df_tokens,
    left_on="judge_name_y",
    right_on="judge_name",
    how="left"
)

# Drop the duplicate judge_name column
df_merged = df_merged.drop(columns=["judge_name"])

# ============ Step 4: Comparison checks ============
print("=== Merged DataFrame Info ===")
print(df_merged.head())
print(f"\nTotal rows: {len(df_merged)}")
print(f"Rows with token data: {df_merged['num_tokens'].notna().sum()}")
print(f"Rows missing token data: {df_merged['num_tokens'].isna().sum()}")

# Check if case counts match
print("\n=== Case Count Comparison ===")
# Group by judge to compare eligible_docs vs num_cases_v2
judge_comparison = df_merged.groupby('judge_name_y').agg({
    'eligible_docs': 'first',
    'num_cases_v2': 'first'
}).reset_index()

judge_comparison['match'] = judge_comparison['eligible_docs'] == judge_comparison['num_cases_v2']
mismatches = judge_comparison[~judge_comparison['match']]

print(f"Judges with matching case counts: {judge_comparison['match'].sum()}")
print(f"Judges with mismatched case counts: {(~judge_comparison['match']).sum()}")

if len(mismatches) > 0:
    print("\nMismatched judges:")
    print(mismatches)

# Check token threshold
df_merged['meets_threshold'] = df_merged['num_tokens'] >= 50000
print(f"\n=== Token Threshold (>=50,000) ===")
print(f"Judges meeting threshold: {df_merged['meets_threshold'].sum()}")
print(f"Judges below threshold: {(~df_merged['meets_threshold']).sum()}")

# ============ Step 5: Save merged file ============
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_test_merged.csv"
df_merged.to_csv(output_path, index=False)
print(f"\n✅ Merged file saved to: {output_path}")

# ============ Step 6: Save filtered version (threshold >= 50k) ============
df_thresh = df_merged[df_merged['num_tokens'] >= 50000].copy()
thresh_output = "/u/home/i/iacir21/myscratch/replication/judge_normalized_test_threshold_50k.csv"
df_thresh.to_csv(thresh_output, index=False)
print(f"✅ Threshold file saved to: {thresh_output}")
print(f"   Judges included: {len(df_thresh['judge_name_y'].unique())}")


=== Merged DataFrame Info ===
              judge_name_y  length_of_judgement case_id  median_year  \
0  margaret waringa muigai                 1067  141502       2016.0   
1             jairus ngaah                 2087  130603       2016.0   
2         pauline nyamweya                 1558  134250       2015.0   
3   maureen atieno onyango                 3296  1582_0       2017.0   
4             oscar angote                  782   858_8       2017.0   

   eligible_docs  train_docs  test_docs  num_tokens  num_cases_v2  
0            367         226        141      199880           141  
1            197         118         79      173926            79  
2            294         172        122      256922           122  
3            164          89         75      160838            75  
4            849         527        322      368199           322  

Total rows: 10764
Rows with token data: 10764
Rows missing token data: 0

=== Case Count Comparison ===
Judges with matching cas

In [ ]:
*******************************************eligible*******************************************************

In [42]:
import pandas as pd

# --- 1. Load data ---
df_eligible = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/eligible_case_ids_vnorm.csv")
df_medians = pd.read_csv("/u/home/i/iacir21/myscratch/train_test_set/median_year_judges_vnorm.csv")

cols_to_keep = ["judge_name_y", "length_of_judgement", "case_id"]
df_final = df_2[cols_to_keep].copy()

# --- 2. Filter df_final to keep only eligible cases ---
df_filtered = df_final[df_final["case_id"].isin(df_eligible["case_id"])].copy()

# --- 3. Merge median year info ---
df_filtered = df_filtered.merge(
    df_medians,
    on="judge_name_y",
    how="left"
)

# --- 4. Save output ---
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible.csv"
df_filtered.to_csv(output_path, index=False)

print(f"✅ Saved eligible judge-level file to:\n{output_path}")
print(f"Rows kept: {len(df_filtered)} / {len(df_final)} total")

# --- 5. Compare judge lists ---
unique_judges_in_filtered = set(df_filtered["judge_name_y"].unique())
unique_judges_in_list = set(list_judge_names)

only_in_list = unique_judges_in_list - unique_judges_in_filtered
only_in_df   = unique_judges_in_filtered - unique_judges_in_list
common       = unique_judges_in_list & unique_judges_in_filtered

print("Only in list_judge_names:", only_in_list)
print("Only in filtered df:", only_in_df)
print("Common:", common)

✅ Saved eligible judge-level file to:
/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible.csv
Rows kept: 26957 / 35395 total
Only in list_judge_names: {'stephen kibunja', 'pk rugut', 'm l nabibya', 's r wewa', 'wilson nkunja kaberia', 's ongeri', 'c n ndegwa', 'b k tanui', 'orenge k i', 'j v o juma', 'cecil henry ethelwood miller', "'s' at nairobi)", 'elena g nderitu', 'temba a sitati', 'r k ondieki', 'm amin j', 'e obaga', 'm wakahora', 'martha a nanzushi', 'mg mugo j', 'norbert okumu', 'etyang a g a j', 'charles gitonga mbogo', 'joseph raymond otieno masime', "a b mong'are", 'stewart mwachiru madzayo', 'jackson kasanga mulwa', 'e malesi', 'l k mutai', 'e muriuki nyagah', 'jane muyoti onyango', 'philip john ransley', 'b p kub0', 'j k sergon', 'kuloba r', 'justus kituku', 'l m wachira', 'john henry sydney todd', 'riaga samuel cornelius omolo', 'raj bahadar bhandari', 'milicent akinyi odeny', 'j kiarie', 'gurbachan singh pall', 's n riechi', 'j m mutungi', 'b m ochoi', 'ja

In [43]:
len(common)

141

In [44]:
print("Common:", common)

Common: {'thande mugure', 'jemutai grace kemei', 'lucy ngima mbugua', 'joel mwaura ngugi', 'alnashir ramazanali magan visram', 'beatrice thuranira jaden', 'jonathan bowen havelock', 'martha karambu koome', 'na', 'george vincent odunga', 'edward muthoga muriithi', 'jessie wanjiku lessit', 'said juma chitembwe', 'roseline lagat-korir', 'byram ongaya', 'daniel kennedy sultani aganyanya', 'david kipyegomen kemei', 'david shikomera majanja', 'nelson jorum abuodha', 'richard mururu mwongo', 'john wycliffe mwera', 'james rika', 'reuben nyambati nyakundi', "anthony ndung'u kimani", "samson odhiambo okong'o", 'kalpana hasmukhrai rawal', 'lesiit jessie w', 'onesmus kimweli mutungi', 'farah s m amin', 'charles yano kimutai', 'patrick j okwaro otieno', 'stella ngali mutuku', 'amraphael mbogholi-msagha', 'john nyabuto onyiego', 'fred andago ochieng', 'james aaron makau', 'john micheal khamoni', 'erastus mwaniki githinji', "hedwig imbosa ong'udi", 'kanyi kimondo', 'radido stephen okiyo', 'thripsisa 

In [45]:
root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_eligible_vnorm"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"

import os
from tqdm import tqdm

for judge in tqdm(list_judge_names):
    
    df_temp = df_filtered[df_filtered["judge_name_y"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
            
        
        f = open(root_path,"r").read()
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        g = open(file_path,"w+")
        g.write(f)
        
# Count how many unique normalized_judge values exist
num_unique_judges = df_final["judge_name_y"].nunique()
print("Number of unique normalized judges:", num_unique_judges)


100% 504/504 [01:58<00:00,  4.24it/s]

Number of unique normalized judges: 141


In [4]:
import os
from tqdm import tqdm
import pandas as pd

root_dir = "/u/home/i/iacir21/myscratch/train_test_set/judgements_txt_eligible"
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"
df_filtered=pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible_v1.csv")
# ============ CONFIG: SET YOUR STARTING JUDGE HERE ============
RESUME_FROM_JUDGE = "yuvinalis maronga angima"  # Replace with actual judge name
SKIP_MODE = True  # Set to False if you want to start fresh from this judge
# ==============================================================

started = False  # Flag to track when to start processing

for judge in tqdm(list_judge_names):
    
    # Skip until we reach the resume point
    if SKIP_MODE:
        if not started:
            if judge == RESUME_FROM_JUDGE:
                started = True
                print(f"\n✅ Starting from judge: {judge}\n")
            else:
                continue  # Skip this judge
    
    df_temp = df_filtered[df_filtered["judge_name_x"] == judge]
    list_ids = list(set(list(df_temp["case_id"])))
    
    print(f"Processing judge: {judge} ({len(list_ids)} cases)")
    
    for case_id in list_ids:
        root_path = os.path.join(root_dir, str(case_id) + ".txt")
        target_path = os.path.join(target_dir, judge)
        
        if not os.path.exists(target_path):
            os.makedirs(target_path)
        
        file_path = os.path.join(target_path, str(case_id) + ".txt")
        
        # Skip if file already exists (resume protection)
        if os.path.exists(file_path):
            continue
            
        try:
            with open(root_path, "r") as f:
                content = f.read()
            
            with open(file_path, "w+") as g:
                g.write(content)
        except Exception as e:
            print(f"Error processing {case_id} for {judge}: {e}")
            continue

print("\n✅ Processing complete!")

# Count how many unique judges
num_unique_judges = df_filtered["judge_name_x"].nunique()
print(f"Number of unique judges: {num_unique_judges}")

  0% 0/504 [00:00<?, ?it/s]


✅ Starting from judge: yuvinalis maronga angima

Processing judge: yuvinalis maronga angima (471 cases)


 99% 500/504 [00:02<00:00, 235.85it/s]

Processing judge: t w murigi (0 cases)
Processing judge: kamau pj j (0 cases)
Processing judge: evans w muleka (0 cases)
Processing judge: joel mwaura ngugi (546 cases)


100% 504/504 [00:02<00:00, 206.89it/s]


✅ Processing complete!
Number of unique judges: 180


In [46]:
import os

# Method 1: Simple and clean (recommended)
target_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"
num_folders = len([name for name in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, name))])
print(f"Number of folders: {num_folders}")

Number of folders: 141


In [47]:
import os
import pandas as pd

# ============ Step 1: Generate token counts (your existing code) ============
base_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"
judge_data = []

for judge_folder in os.listdir(base_dir):
    judge_path = os.path.join(base_dir, judge_folder)
    
    if os.path.isdir(judge_path):
        total_words = 0
        num_cases_v2 = 0
        
        for file_name in os.listdir(judge_path):
            if file_name.endswith(".txt"):
                file_path = os.path.join(judge_path, file_name)
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                    total_words += len(text.split())
                    num_cases_v2 += 1
        
        judge_data.append([judge_folder, total_words, num_cases_v2])

df_tokens = pd.DataFrame(judge_data, columns=["judge_name", "num_tokens", "num_cases_v2"])

# ============ Step 2: Load existing CSV ============
df_existing = pd.read_csv("/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible.csv")

# ============ Step 3: Merge the two dataframes ============
df_merged = df_existing.merge(
    df_tokens,
    left_on="judge_name_y",
    right_on="judge_name",
    how="left"
)

# Drop the duplicate judge_name column
df_merged = df_merged.drop(columns=["judge_name"])

# ============ Step 4: Comparison checks ============
print("=== Merged DataFrame Info ===")
print(df_merged.head())
print(f"\nTotal rows: {len(df_merged)}")
print(f"Rows with token data: {df_merged['num_tokens'].notna().sum()}")
print(f"Rows missing token data: {df_merged['num_tokens'].isna().sum()}")

# Check if case counts match
print("\n=== Case Count Comparison ===")
# Group by judge to compare eligible_docs vs num_cases_v2
judge_comparison = df_merged.groupby('judge_name_y').agg({
    'eligible_docs': 'first',
    'num_cases_v2': 'first'
}).reset_index()

judge_comparison['match'] = judge_comparison['eligible_docs'] == judge_comparison['num_cases_v2']
mismatches = judge_comparison[~judge_comparison['match']]

print(f"Judges with matching case counts: {judge_comparison['match'].sum()}")
print(f"Judges with mismatched case counts: {(~judge_comparison['match']).sum()}")

if len(mismatches) > 0:
    print("\nMismatched judges:")
    print(mismatches)

# Check token threshold
df_merged['meets_threshold'] = df_merged['num_tokens'] >= 50000
print(f"\n=== Token Threshold (>=50,000) ===")
print(f"Judges meeting threshold: {df_merged['meets_threshold'].sum()}")
print(f"Judges below threshold: {(~df_merged['meets_threshold']).sum()}")

# ============ Step 5: Save merged file ============
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible_merged.csv"
df_merged.to_csv(output_path, index=False)
print(f"\n✅ Merged file saved to: {output_path}")

# ============ Step 6: Save filtered version (threshold >= 50k) ============
df_thresh = df_merged[df_merged['num_tokens'] >= 50000].copy()
thresh_output = "/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible_threshold_50k.csv"
df_thresh.to_csv(thresh_output, index=False)
print(f"✅ Threshold file saved to: {thresh_output}")
print(f"   Judges included: {len(df_thresh['judge_name_y'].unique())}")


=== Merged DataFrame Info ===
              judge_name_y  length_of_judgement case_id  median_year  \
0   anthony ndung'u kimani                  710  139782       2017.0   
1         pauline nyamweya                  972   83775       2015.0   
2  margaret waringa muigai                 1067  141502       2016.0   
3   joseph kiplagat sergon                  186   49978       2013.0   
4             jairus ngaah                 2087  130603       2016.0   

   eligible_docs  train_docs  test_docs  num_tokens  num_cases_v2  
0            153          79         74      221392           153  
1            294         172        122      580304           294  
2            367         226        141      543246           367  
3            777         412        365      835811           777  
4            197         118         79      406565           197  

Total rows: 26957
Rows with token data: 26957
Rows missing token data: 0

=== Case Count Comparison ===
Judges with matching cas

In [49]:
import os
import pandas as pd

# --- Paths ---
csv_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_test.csv"
corpus_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_test"
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_test_v2.csv"

# --- Load CSV ---
df = pd.read_csv(csv_path)

# --- Get folder names in judge_corpus_test ---
folders = [
    f for f in os.listdir(corpus_dir)
    if os.path.isdir(os.path.join(corpus_dir, f))
]
print(f"Number of folders in judge_corpus_test: {len(folders)}")

# --- Unique judges in CSV ---
unique_judges = df['judge_name_y'].unique()
print(f"Number of unique judges in CSV: {len(unique_judges)}")

# --- Judges present in CSV but missing as folders ---
missing_judges = set(unique_judges) - set(folders)
print(f"Judges being dropped (no folder found): {missing_judges}")


Number of folders in judge_corpus_test: 141
Number of unique judges in CSV: 141
Judges being dropped (no folder found): set()


In [2]:

# --- Filter and save ---
df_v2 = df[df['judge_name_x'].isin(folders)]
print(f"Rows before: {len(df)} | Rows after: {len(df_v2)}")

df_v2.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Rows before: 59977 | Rows after: 59556
Saved to /u/home/i/iacir21/myscratch/replication/judge_normalized_test_v2.csv


In [51]:
import os
import pandas as pd

# --- Paths ---
csv_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_train.csv"
corpus_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_train"
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_train_v2.csv"

# --- Load CSV ---
df = pd.read_csv(csv_path)

# --- Get folder names in judge_corpus_test ---
folders = [
    f for f in os.listdir(corpus_dir)
    if os.path.isdir(os.path.join(corpus_dir, f))
]
print(f"Number of folders in judge_corpus_test: {len(folders)}")

# --- Unique judges in CSV ---
unique_judges = df['judge_name_y'].unique()
print(f"Number of unique judges in CSV: {len(unique_judges)}")

# --- Judges present in CSV but missing as folders ---
missing_judges = set(unique_judges) - set(folders)
print(f"Judges being dropped (no folder found): {missing_judges}")

Number of folders in judge_corpus_test: 141
Number of unique judges in CSV: 141
Judges being dropped (no folder found): set()


In [52]:

# --- Filter and save ---
df_v2 = df[df['judge_name_x'].isin(folders)]
print(f"Rows before: {len(df)} | Rows after: {len(df_v2)}")

df_v2.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

KeyError: 'judge_name_x'

In [53]:
import os
import pandas as pd

# --- Paths ---
csv_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible.csv"
corpus_dir = "/u/home/i/iacir21/myscratch/replication/judges_corpus_eligible"
output_path = "/u/home/i/iacir21/myscratch/replication/judge_normalized_eligible_v2.csv"

# --- Load CSV ---
df = pd.read_csv(csv_path)

# --- Get folder names in judge_corpus_test ---
folders = [
    f for f in os.listdir(corpus_dir)
    if os.path.isdir(os.path.join(corpus_dir, f))
]
print(f"Number of folders in judge_corpus_test: {len(folders)}")

# --- Unique judges in CSV ---
unique_judges = df['judge_name_y'].unique()
print(f"Number of unique judges in CSV: {len(unique_judges)}")

# --- Judges present in CSV but missing as folders ---
missing_judges = set(unique_judges) - set(folders)
print(f"Judges being dropped (no folder found): {missing_judges}")

Number of folders in judge_corpus_test: 141
Number of unique judges in CSV: 141
Judges being dropped (no folder found): set()


In [7]:

# --- Filter and save ---
df_v2 = df[df['judge_name_x'].isin(folders)]
print(f"Rows before: {len(df)} | Rows after: {len(df_v2)}")

df_v2.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Rows before: 137713 | Rows after: 136587
Saved to /u/home/i/iacir21/myscratch/replication/judge_normalized_eligible_v2.csv
